# 08. Gesture Classification via Frequency Sparse Features
**Objective:** Evaluate if K-SVD and Mini-Batch sparse features extracted from the Frequency Domain can accurately classify gestures on Subject 1.

In [1]:
%load_ext autoreload
%autoreload 2

import sys
import os
import h5py
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler

sys.path.append(os.path.abspath('../'))
from src.config import PREPROCESSED_DIR, DEVICE
from src.frequency import extract_fft_magnitude
from src.dictionary_freq import FrequencyDictionaryLearner, FrequencyOMPExtractor
from src.classification import train_svm_classifier, evaluate_classifier

plt.style.use('seaborn-v0_8-whitegrid')

### 1. Load Subject 1 Data & Prepare Frequency Transformation

In [2]:
file_path = os.path.join(PREPROCESSED_DIR, "DB1_subject_1.h5")
with h5py.File(file_path, 'r') as f:
    X_bal = np.array(f['X'])
    y_bal = np.array(f['y']).astype(np.int64)
    reps_bal = np.array(f['reps'])

train_reps = [1, 2, 3, 4, 5, 6, 7]
test_reps = [8, 9, 10]

train_idx = np.where(np.isin(reps_bal, train_reps))[0]
test_idx = np.where(np.isin(reps_bal, test_reps))[0]

# Transform whole dataset to Frequency Domain
X_freq = extract_fft_magnitude(X_bal)

X_freq_train, y_train = X_freq[train_idx], y_bal[train_idx]
X_freq_test, y_test = X_freq[test_idx], y_bal[test_idx]

Extracting FFT from input shape (18630, 20, 10)...
FFT extraction complete. Output shape: (18630, 10, 10)


### 2. Extract Sparse Codes (K-SVD & Mini-Batch)

In [3]:
ksvd_learner = FrequencyDictionaryLearner(n_atoms=32, transform_n_nonzero_coefs=3, method='ksvd')
ksvd_learner.load_dictionary("emg_dict_ksvd_freq_S1.npz")
ksvd_extractor = FrequencyOMPExtractor(ksvd_learner.dictionary_, n_nonzero_coefs=3)

mb_learner = FrequencyDictionaryLearner(n_atoms=32, transform_n_nonzero_coefs=3, method='minibatch')
mb_learner.load_dictionary("emg_dict_minibatch_freq_S1.npz")
mb_extractor = FrequencyOMPExtractor(mb_learner.dictionary_, n_nonzero_coefs=3)

# Extract Features
X_train_ksvd = ksvd_extractor.transform(X_freq_train)
X_test_ksvd = ksvd_extractor.transform(X_freq_test)

X_train_mb = mb_extractor.transform(X_freq_train)
X_test_mb = mb_extractor.transform(X_freq_test)

# Standardize Sparse Matrices
scaler_ksvd = StandardScaler()
X_train_ksvd_scaled = scaler_ksvd.fit_transform(X_train_ksvd)
X_test_ksvd_scaled = scaler_ksvd.transform(X_test_ksvd)

scaler_mb = StandardScaler()
X_train_mb_scaled = scaler_mb.fit_transform(X_train_mb)
X_test_mb_scaled = scaler_mb.transform(X_test_mb)

Dictionary loaded from /workspaces/TCC/models/emg_dict_ksvd_freq_S1.npz
Dictionary loaded from /workspaces/TCC/models/emg_dict_minibatch_freq_S1.npz


/home/vscode/.local/lib/python3.12/site-packages/sklearn/utils/_param_validation.py:191: RuntimeWarning: Orthogonal matching pursuit ended prematurely due to linear dependence in the dictionary. The requested precision might not have been met.
  return func(*args, **kwargs)
/home/vscode/.local/lib/python3.12/site-packages/sklearn/utils/_param_validation.py:191: RuntimeWarning: Orthogonal matching pursuit ended prematurely due to linear dependence in the dictionary. The requested precision might not have been met.
  return func(*args, **kwargs)
/home/vscode/.local/lib/python3.12/site-packages/sklearn/utils/_param_validation.py:191: RuntimeWarning: Orthogonal matching pursuit ended prematurely due to linear dependence in the dictionary. The requested precision might not have been met.
  return func(*args, **kwargs)
/home/vscode/.local/lib/python3.12/site-packages/sklearn/utils/_param_validation.py:191: RuntimeWarning: Orthogonal matching pursuit ended prematurely due to linear dependence

### 3. Benchmark A: Support Vector Machine (SVM)

In [4]:
print("--- Training SVM on K-SVD Frequency Features ---")
svm_ksvd = train_svm_classifier(X_train_ksvd_scaled, y_train, kernel="rbf", C=1.0)
results_ksvd_svm = evaluate_classifier(svm_ksvd, X_test_ksvd_scaled, y_test)

print(f"K-SVD + SVM Accuracy: {results_ksvd_svm['accuracy'] * 100:.2f}%")
print(f"K-SVD + SVM Macro F1: {results_ksvd_svm['macro_f1'] * 100:.2f}%\n")

print("--- Training SVM on Mini-Batch Frequency Features ---")
svm_mb = train_svm_classifier(X_train_mb_scaled, y_train, kernel="rbf", C=1.0)
results_mb_svm = evaluate_classifier(svm_mb, X_test_mb_scaled, y_test)

print(f"Mini-Batch + SVM Accuracy: {results_mb_svm['accuracy'] * 100:.2f}%")
print(f"Mini-Batch + SVM Macro F1: {results_mb_svm['macro_f1'] * 100:.2f}%")

--- Training SVM on K-SVD Frequency Features ---
Training SVM (Kernel: rbf) on 12664 samples...
Training complete.
Evaluating model on test set...
K-SVD + SVM Accuracy: 16.73%
K-SVD + SVM Macro F1: 15.54%

--- Training SVM on Mini-Batch Frequency Features ---
Training SVM (Kernel: rbf) on 12664 samples...
Training complete.
Evaluating model on test set...
Mini-Batch + SVM Accuracy: 17.08%
Mini-Batch + SVM Macro F1: 15.92%


### 4. Benchmark B: Deep Neural Network Classifier (PyTorch MLP)

In [5]:
class SparseCodeClassifier(nn.Module):
    def __init__(self, input_dim: int, num_classes: int):
        super(SparseCodeClassifier, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes)
        )
        
    def forward(self, x):
        return self.net(x)

def train_sparse_nn(X_tr_s, y_tr_s, X_te_s, y_te_s, epochs=30):
    num_classes = int(max(y_tr_s.max(), y_te_s.max()) + 1)
    input_dim = X_tr_s.shape[1]
    
    model = SparseCodeClassifier(input_dim, num_classes).to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)
    
    train_loader = DataLoader(
        TensorDataset(torch.FloatTensor(X_tr_s), torch.LongTensor(y_tr_s)), 
        batch_size=64, 
        shuffle=True
    )
    
    for epoch in range(epochs):
        model.train()
        for bx, by in train_loader:
            bx, by = bx.to(DEVICE), by.to(DEVICE)
            optimizer.zero_grad()
            out = model(bx)
            loss = criterion(out, by)
            loss.backward()
            optimizer.step()
            
    model.eval()
    with torch.no_grad():
        test_x = torch.FloatTensor(X_te_s).to(DEVICE)
        preds = torch.argmax(model(test_x), dim=1).cpu().numpy()
        
    acc = np.mean(preds == y_te_s)
    return acc

print("\n--- Training PyTorch Deep Classifier on K-SVD Sparse Features ---")
ksvd_nn_acc = train_sparse_nn(X_train_ksvd_scaled, y_train, X_test_ksvd_scaled, y_test)
print(f"K-SVD + Neural Network Accuracy: {ksvd_nn_acc * 100:.2f}%")

print("\n--- Training PyTorch Deep Classifier on Mini-Batch Sparse Features ---")
mb_nn_acc = train_sparse_nn(X_train_mb_scaled, y_train, X_test_mb_scaled, y_test)
print(f"Mini-Batch + Neural Network Accuracy: {mb_nn_acc * 100:.2f}%")


--- Training PyTorch Deep Classifier on K-SVD Sparse Features ---
K-SVD + Neural Network Accuracy: 39.27%

--- Training PyTorch Deep Classifier on Mini-Batch Sparse Features ---
Mini-Batch + Neural Network Accuracy: 39.81%
